# CodeGen agent tests

Tests the parser → formulator → codegen chain, runs the generated script,
then checks codegen's error-handling branches without calling the LLM.
Run cells top to bottom.

**Setup:** ensure `.env` has `ANTHROPIC_API_KEY` (copy from `.env.example`).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
if not (project_root / "orharness").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not set. Copy .env.example to .env and add your key."
    )

print("API key loaded from .env")

In [ ]:
from orharness.models import ORHarnessConfig
from orharness.agents.parser import parse_problem
from orharness.agents.formulator import formulate_problem
from orharness.agents.codegen import generate_code, RESULT_MARKER

config = ORHarnessConfig()
print(config)

## Test: scheduling problem (parser → formulator → codegen)

Generate an OR-Tools script for the nurse problem. Expect a CP-SAT solver
and syntactically valid, self-contained Python.

In [ ]:
parsed = parse_problem(
    "I have 6 nurses, 3 shifts per day, 7 days a week. "
    "No nurse works more than 5 shifts per week. "
    "Night shifts need at least 2 nurses.",
    config,
)
formulated = formulate_problem(parsed, config)
generated = generate_code(formulated, config)

print("solver:", generated.solver)
print("problem_type:", generated.problem_type)
print("attempt:", generated.attempt)
print("code length:", len(generated.code), "chars")
print("\n--- generated code ---\n")
print(generated.code)

## Run the generated script

Execute it in a subprocess (a stand-in for the real sandbox, which doesn't
exist yet), then split on the result marker and parse the JSON the way the
pipeline will.

In [ ]:
import json
import subprocess
import tempfile

with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
    f.write(generated.code)
    script_path = f.name

proc = subprocess.run(
    ["python3", script_path],
    capture_output=True,
    text=True,
    timeout=config.timeout_seconds,
)

print("exit code:", proc.returncode)
if proc.stderr:
    print("stderr:", proc.stderr[:500])

if RESULT_MARKER in proc.stdout:
    payload = proc.stdout.split(RESULT_MARKER, 1)[1].strip()
    result = json.loads(payload)
    print("\nparsed result:")
    print("  status:", result["status"])
    print("  feasible:", result["feasible"])
    print("  objective_value:", result["objective_value"])
    print("  solve_time_seconds:", result["solve_time_seconds"])
    print("  solution keys:", list(result["solution"].keys()))
else:
    print("\nNo result marker found in stdout")

## Test: codegen error handling (no LLM)

Force each failure branch in `generate_code` by swapping the LLM call
(`completion`) for a fake that returns canned bad code. Plus one branch
(unmapped problem type) that fails before any LLM call. Each must raise
`CodeGenerationError`.

In [ ]:
from types import SimpleNamespace

import orharness.agents.codegen as cgmod
from orharness.models import FormulatedModel, ProblemType
from orharness.exceptions import CodeGenerationError

dummy_formulated = FormulatedModel(
    problem_type=ProblemType.SCHEDULING,
    variables=["x[i]: binary"],
    objective="find feasible solution",
    constraints=["c1"],
    parameters={},
)


def fake_completion_returning(content: str):
    def _fake(*args, **kwargs):
        message = SimpleNamespace(content=content)
        return SimpleNamespace(choices=[SimpleNamespace(message=message)])
    return _fake


def expect_codegen_error(label: str, fake_content: str):
    cgmod.completion = fake_completion_returning(fake_content)
    try:
        cgmod.generate_code(dummy_formulated, config)
    except CodeGenerationError as e:
        print(f"PASS [{label}] raised CodeGenerationError: {str(e)[:60]}...")
    else:
        print(f"FAIL [{label}] expected CodeGenerationError, none raised")


original_completion = cgmod.completion
try:
    expect_codegen_error("empty code", "")
    expect_codegen_error("syntax error", "from ortools.sat.python import cp_model\ndef (")
    expect_codegen_error("missing ortools", "x = 1\nprint(x)")

    # Unmapped problem type fails before the LLM is ever called.
    unknown_formulated = dummy_formulated.model_copy(
        update={"problem_type": ProblemType.UNKNOWN}
    )
    try:
        cgmod.generate_code(unknown_formulated, config)
    except CodeGenerationError as e:
        print(f"PASS [unmapped type] raised CodeGenerationError: {str(e)[:60]}...")
    else:
        print("FAIL [unmapped type] expected CodeGenerationError, none raised")

    # Positive control: valid code should return a GeneratedCode.
    cgmod.completion = fake_completion_returning(
        "from ortools.sat.python import cp_model\nprint('hello')"
    )
    ok = cgmod.generate_code(dummy_formulated, config)
    print(f"PASS [valid code] -> solver={ok.solver}, {len(ok.code)} chars")
finally:
    cgmod.completion = original_completion  # restore the real LLM call